# 🐍 Módulo 00 · Python para Ciencia de Datos

El Python que necesitás para leer, escribir, modificar y debuggear código de datos. La base de todo lo que sigue.

Versión web con conceptos, reconocimiento, razonamiento y proyecto: [https://rosalesluciano.github.io/datalab/](https://rosalesluciano.github.io/datalab/#/m/m00)

## Cómo trabajar este notebook
1. Ejecutá la celda ⚙️ (carga datos y verificador).
2. **Antes de escribir código**, escribí en un comentario cómo lo vas a resolver.
3. Implementá y guardá tu respuesta en `resultado` (o lo que pida el ejercicio).
4. `comprobar()` verifica. Si te trabás: `ver_pista("id", 1)`, después 2 y 3.
5. `ver_solucion("id")` es el último recurso: muestra idea → pseudocódigo → código.

## Conceptos del módulo
- **Variables y tipos de datos**: Guardar un dato con un nombre para reutilizarlo, y saber **qué tipo** de dato es (número, texto, verdadero/falso, vacío). El tipo decide qué operaciones tienen sentido.
- **Números y operaciones**: Calcular métricas: totales, promedios, porcentajes, variaciones, descuentos, impuestos.
- **Strings (texto)**: Limpiar, normalizar, separar y armar texto: nombres escritos a mano, códigos, líneas de un archivo, reportes.
- **Listas**: Guardar una **colección ordenada** de valores (ventas de cada día, lecturas de un sensor) para recorrerla, resumirla o recortarla.
- **Tuplas y sets**: **Tupla**: agrupar valores que van juntos y no deberían cambiar (un registro, varios resultados de una función). **Set**: quedarte con valores únicos y preguntar pertenencia rápido.
- **Diccionarios**: Buscar un valor **por su nombre** (clave) en lugar de por posición: precio por producto, total por cliente. También representar un registro con campos con nombre.
- **Booleanos y comparaciones**: Hacer **preguntas** sobre los datos que se responden sí/no: ¿supera el umbral?, ¿es premium y de España?, ¿el producto está en el catálogo?
- **Condicionales**: Tomar **decisiones** distintas según los datos: clasificar montos en tramos, aplicar reglas de negocio, descartar valores inválidos.
- **Loop for**: Repetir la misma operación **para cada elemento** de una colección: calcular el ingreso de cada venta, acumular totales, construir resultados.
- **Loop while**: Repetir **mientras** se cumpla una condición, cuando no sabés de antemano cuántas vueltas hacen falta.
- **Funciones: parámetros y return**: **Empaquetar** una lógica con un nombre para reutilizarla, probarla por separado y cambiarla en un solo lugar.
- **Comprensiones**: Construir una lista o diccionario nuevo **transformando y/o filtrando** otro, en una línea legible.
- **Errores y excepciones**: **Leer** qué salió mal (el traceback te dice tipo de error y línea) y **manejar** datos inválidos sin que el programa se caiga.
- **Módulos e imports**: Usar código que ya existe (de Python o de otros) en lugar de escribirlo desde cero: estadística, fechas, archivos, pandas, NumPy.
- **Trabajar con archivos**: Leer datos guardados en archivos (CSV, JSON, texto) y escribir resultados.
- **Leer código y debuggear**: Entender qué hace un código (tuyo, de otra persona o de una IA) y encontrar por qué no hace lo que esperás.
- **Python aplicado a datos**: Responder preguntas de datos (totales por grupo, rankings, filtros) combinando las piezas de Python. Es exactamente lo que después pandas hace en una línea.

In [ ]:
#@title ⚙️ 1) Ejecutá esta celda primero (datos y verificador) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series", list: "una lista", dict: "un diccionario",
               tuple: "una tupla", set: "un set", str: "un texto (str)", int: "un entero (int)",
               float: "un decimal (float)", bool: "un booleano (bool)", type(None): "None (nada)",
               np.ndarray: "un array de NumPy"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    if isinstance(e, dict):
        return (isinstance(u, dict) and set(map(str, u)) == set(map(str, e))
                and all(_scalar_eq(u[k] if k in u else u[str(k)], v) for k, v in e.items()))
    if isinstance(e, (set, frozenset)):
        return isinstance(u, (set, frozenset)) and u == e
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    if u is None:
        return False, ("Tu resultado es None. ¿Te falta un `return` en la función, o guardaste lo que devuelve "
                       "un método que modifica en el lugar (como `.sort()` o `.append()`)?")
    contenedores = (list, tuple, dict, set, str)
    if isinstance(e, contenedores) and not isinstance(u, type(e)) and not (
            isinstance(e, (list, tuple)) and isinstance(u, (list, tuple))):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}."
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple)) and len(u) != len(e):
        return False, f"Tu resultado tiene {len(u)} elementos y se esperaban {len(e)}."
    if isinstance(e, dict) and isinstance(u, dict) and set(map(str, u)) != set(map(str, e)):
        faltan = [k for k in e if str(k) not in set(map(str, u))]
        sobran = [k for k in u if str(k) not in set(map(str, e))]
        msg = "Las claves del diccionario no coinciden."
        if faltan:
            msg += f" Faltan: {faltan}."
        if sobran:
            msg += f" Sobran: {sobran}."
        return False, msg
    txt = repr(u)
    return False, f"Tu resultado es {txt[:300] + ('…' if len(txt) > 300 else '')}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usá & (y), | (o) y poné cada condición entre paréntesis."),
    ("can't multiply sequence", "Estás multiplicando un texto (str). ¿El número llegó como texto? Convertilo con int() o float()."),
    ("can only concatenate str", "Estás sumando texto con otra cosa. Convertí con str() para unir textos, o con float() para hacer cuentas."),
    ("unsupported operand type", "Los dos lados de la operación tienen tipos incompatibles (por ejemplo, int + str o None + float). Revisá type() de cada valor."),
    ("could not convert string to float", "Hay un texto que no representa un número (por ejemplo 'n/d' o ''). Filtralo o capturá el ValueError con try/except."),
    ("invalid literal for int", "Hay un texto que no representa un entero. Revisá el valor o usá try/except ValueError."),
    ("index out of range", "Pediste una posición que no existe. Recordá: las posiciones van de 0 a len()-1, y -1 es el último."),
    ("values to unpack", "La cantidad de variables a la izquierda no coincide con la cantidad de valores que llegan."),
    ("ZeroDivisionError", "Dividiste por cero. ¿La lista o el divisor pueden estar vacíos? Verificalo antes de dividir."),
    ("'NoneType' object", "Un valor es None. Suele pasar con una función sin return o con métodos como .sort() que devuelven None."),
    ("KeyError", "Esa clave o columna no existe. Revisá mayúsculas, tildes y espacios; con dicts podés usar .get()."),
    ("NameError", "Usás una variable que no existe (todavía). ¿Error de tipeo o la definiste más abajo?"),
    ("IndentationError", "Revisá la sangría: el contenido de if, for, def, etc. va con 4 espacios."),
    ("SyntaxError", "Error de sintaxis: revisá paréntesis, comillas, dos puntos (:) al final de if/for/def y que cada línea esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este tipo de objeto. Revisá el nombre y qué tipo tiene la variable."),
    ("MergeError", "Revisá las columnas que usás para unir (on / left_on / right_on)."),
    ("TypeError", "Algún tipo de dato no encaja con la operación. Revisá type() de las variables involucradas."),
    ("ValueError", "El tipo es correcto pero el valor no es válido para esta operación."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import os, json, tempfile\n\n# Ventas de una tienda: cada diccionario es una fila\nventas = [\n    {"id": 1, "fecha": "2024-03-01", "producto": "Laptop", "categoria": "Computación", "cantidad": 1, "precio": 1200.0, "cliente": "Ana"},\n    {"id": 2, "fecha": "2024-03-01", "producto": "Mouse", "categoria": "Accesorios", "cantidad": 3, "precio": 25.5, "cliente": "Bruno"},\n    {"id": 3, "fecha": "2024-03-02", "producto": "Monitor", "categoria": "Computación", "cantidad": 2, "precio": 300.0, "cliente": "Carla"},\n    {"id": 4, "fecha": "2024-03-02", "producto": "Auriculares", "categoria": "Audio", "cantidad": 1, "precio": 80.0, "cliente": "Ana"},\n    {"id": 5, "fecha": "2024-03-03", "producto": "Teclado", "categoria": "Accesorios", "cantidad": 2, "precio": 45.0, "cliente": "Diego"},\n    {"id": 6, "fecha": "2024-03-03", "producto": "Laptop", "categoria": "Computación", "cantidad": 2, "precio": 1200.0, "cliente": "Elena"},\n    {"id": 7, "fecha": "2024-03-04", "producto": "Mouse", "categoria": "Accesorios", "cantidad": 5, "precio": 25.5, "cliente": "Carla"},\n    {"id": 8, "fecha": "2024-03-04", "producto": "Micrófono", "categoria": "Audio", "cantidad": 1, "precio": 95.0, "cliente": "Bruno"},\n    {"id": 9, "fecha": "2024-03-05", "producto": "Tablet", "categoria": "Computación", "cantidad": 1, "precio": 450.0, "cliente": "Ana"},\n    {"id": 10, "fecha": "2024-03-05", "producto": "Auriculares", "categoria": "Audio", "cantidad": 3, "precio": 80.0, "cliente": "Diego"},\n]\n\n# Catálogo: producto -> precio\nprecios = {"Laptop": 1200.0, "Mouse": 25.5, "Teclado": 45.0, "Monitor": 300.0,\n           "Auriculares": 80.0, "Tablet": 450.0, "Micrófono": 95.0}\n\n# Ventas totales de cada día de una semana (lun a dom)\nventas_semana = [1200, 950, 1430, 880, 1700, 2100, 1900]\n\n# Lecturas de un sensor de temperatura: None = sin lectura, -999.0 = error del sensor\nlecturas = [21.5, 22.1, None, 23.4, -999.0, 22.8, None, 24.0, 85.3, 23.1]\n\n# Archivos de ejemplo (rutas guardadas en variables, como en un proyecto real)\nRUTA_VENTAS = os.path.join(tempfile.gettempdir(), "pq_ventas.csv")\nwith open(RUTA_VENTAS, "w", encoding="utf-8") as _f:\n    _f.write("""fecha,producto,cantidad,precio\n2024-03-01,Laptop,1,1200\n2024-03-01,Mouse,3,25.5\n2024-03-02,Monitor,2,300\n2024-03-02,Mouse,1,25.5\n2024-03-03,Teclado,2,45\n""")\n\nRUTA_VENTAS_SUCIO = os.path.join(tempfile.gettempdir(), "pq_ventas_sucio.csv")\nwith open(RUTA_VENTAS_SUCIO, "w", encoding="utf-8") as _f:\n    _f.write("""fecha,producto,cantidad,precio\n2024-03-01,Laptop,1,1200\n2024-03-01,Mouse,3,25.5\n2024-03-02,Monitor,n/d,300\n2024-03-02,Auriculares,2,80\n2024-03-03,Teclado,,45\n2024-03-03,Laptop,2,1200\n2024-03-04,Mouse,-1,25.5\n2024-03-04,Tablet,1,450\n""")\n\nRUTA_CONFIG = os.path.join(tempfile.gettempdir(), "pq_config.json")\nwith open(RUTA_CONFIG, "w", encoding="utf-8") as _f:\n    json.dump({"moneda": "USD", "iva": 0.21, "umbral_alerta": 1000, "tiendas": ["Centro", "Norte"]}, _f)\ndel _f\n'
_SOL_B64 = '...'
_TABLAS = ['ventas', 'precios', 'ventas_semana', 'lecturas']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todos los datos limpios (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    from IPython.display import HTML, display
    import html as _h
    if ok:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">✅ Verificado. Ahora explicá con tus palabras por qué funciona '
                     'y probá una variante.</div>'))
    else:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_pista(ej, nivel=1):
    """Pistas progresivas (1-3). Pedí la siguiente solo si la anterior no alcanzó."""
    pistas = _SOL[ej]["hints"]
    print(f"Pista {nivel}/{len(pistas)}: {pistas[min(nivel, len(pistas)) - 1]}")

def ver_solucion(ej):
    """Último recurso: primero la idea, después el pseudocódigo y al final el código."""
    info = _SOL[ej]
    print("💡 Idea:\n" + info["explain"] + "\n")
    print("🧭 Pseudocódigo:\n" + info["pseudo"] + "\n")
    print("🧑‍💻 Código:\n" + info["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 Variables, tipos y números

## 1. ¿Cuánto paga el cliente?
`Código` · `Fácil`

Un cliente compró **3 mouse a $25.5** cada uno. Tiene un **10% de descuento** sobre el subtotal. Calculá cuánto paga en total y guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

precio_unitario = 25.5
cantidad = 3
descuento = 0.10

resultado = 

comprobar("py-d01")

## 2. El total que no tiene sentido
`Debugging` · `Fácil`

Este código debería calcular el total de 2 unidades de un producto, pero el resultado es rarísimo. Ejecutalo, mirá qué devuelve y corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

precio = "1200.50"   # llegó así desde un formulario
cantidad = 2

resultado = precio * cantidad

comprobar("py-e01")

---
# 📌 Strings: texto que llega sucio

## 3. Normalizar un nombre
`Código` · `Fácil`

El nombre de una clienta llegó como `'  carla GÓMEZ '`. Dejalo como `'Carla Gómez'` (sin espacios sobrantes, cada palabra con mayúscula inicial) y guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

nombre = "  carla GÓMEZ "

resultado = 

comprobar("py-d02")

## 4. Una línea de reporte
`Código` · `Medio`

Armá el texto de una línea de reporte con el formato exacto `'Monitor: $1,234.50'` a partir de las variables `producto` y `total` (separador de miles con coma y siempre 2 decimales). Guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

producto = "Monitor"
total = 1234.5

resultado = 

comprobar("py-d03")

## 5. La IA parte una línea de CSV
`IA copiloto` · `Medio`

Necesitás calcular el ingreso de una línea de texto tomada de un CSV: `'2024-03-01,Laptop,1,1200'` (fecha, producto, cantidad, precio). Le pediste ayuda a una IA y respondió con el código de abajo. **Antes de usarlo, tenés que entenderlo.**

> 🤖 Prompt usado:
> Tengo una línea de texto de un CSV: '2024-03-01,Laptop,1,1200' (fecha, producto, cantidad, precio). Quiero calcular cantidad × precio en Python puro. Mostrame el código y explicame por qué cada conversión es necesaria.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

linea = "2024-03-01,Laptop,1,1200"

partes = linea.split(",")
fecha, producto, cantidad, precio = partes
resultado = int(cantidad) * float(precio)

comprobar("py-f01")

---
# 📌 Listas, tuplas y sets

## 6. Resumen de la semana
`Código` · `Fácil`

Con `ventas_semana` (ya cargada: ventas de lunes a domingo), armá un resumen como **lista** con tres valores: `[máximo, mínimo, promedio redondeado a 2 decimales]`. Guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

print(ventas_semana)

resultado = 

comprobar("py-d04")

## 7. Solo el fin de semana
`Código` · `Fácil`

Quedate con las ventas del **fin de semana** (los dos últimos días de `ventas_semana`) y guardalas en `resultado` como lista. Tu código debería seguir funcionando aunque la lista tuviera más semanas al principio.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("py-d05")

## 8. El último día que no existe
`Debugging` · `Fácil`

Este código quiere obtener la venta del último día, pero falla. Encontrá el problema y corregilo **sin escribir el número de posición a mano**.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

ultimo = ventas_semana[len(ventas_semana)]
resultado = ultimo

comprobar("py-e02")

## 9. Categorías únicas y ordenadas
`Código` · `Fácil`

A partir de `categorias_vendidas`, obtené la **lista** de categorías distintas **en orden alfabético**. Guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

categorias_vendidas = ["Audio", "Computación", "Audio", "Accesorios", "Computación", "Muebles", "Accesorios"]

resultado = 

comprobar("py-d06")

---
# 📌 Diccionarios

## 10. Actualizar el catálogo
`Código` · `Fácil`

Actualizá el diccionario `precios`: agregá el producto `'Webcam'` a **60.0** y subí el precio del `'Mouse'` un **10%**. Guardá el diccionario actualizado en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

print(precios)

resultado = precios

comprobar("py-d07")

## 11. El precio que no aparece
`Debugging` · `Fácil`

Este código calcula el costo de 3 mouse, pero se rompe. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

costo = precios["mouse"] * 3
resultado = costo

comprobar("py-e03")

---
# 📌 Decisiones: booleanos y condicionales

## 12. Clasificar montos en tramos
`Código` · `Medio`

Completá la función `clasificar_monto(monto)` para que devuelva:
- `'alta'` si el monto es **500 o más**,
- `'media'` si es **100 o más** (y menos de 500),
- `'baja'` en otro caso.

Se va a probar con varios montos, incluidos los bordes.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

def clasificar_monto(monto):
    # tu lógica aquí
    return "baja"

comprobar("py-d08")

## 13. La rama que nunca se ejecuta
`Debugging` · `Medio`

Esta función debería devolver `'muy alta'` para montos de más de 1000, pero nunca lo hace. Corregila.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

def nivel(monto):
    if monto > 500:
        return "alta"
    elif monto > 1000:
        return "muy alta"
    else:
        return "normal"

comprobar("py-e04")

---
# 📌 Loops: repetir sobre los datos

## 14. Total facturado
`Código` · `Fácil`

Calculá el **total facturado** (suma de `cantidad × precio` de cada venta) de la lista `ventas` y guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

print(ventas[0])   # mirá cómo es una venta

resultado = 

comprobar("py-d09")

## 15. Ventas por categoría
`Código` · `Medio`

Implementá el plan del ejercicio de planificación: contá **cuántas ventas** hay de cada `categoria` en `ventas`. Guardá un diccionario `{categoría: cantidad_de_ventas}` en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

conteo = {}
# recorré las ventas y contá

resultado = conteo

comprobar("py-d10")

## 16. Transferencia: unidades por categoría
`Transferencia` · `Medio`

Calculá la suma de `cantidad` por `categoria` en `ventas`. Guardá el diccionario `{categoría: unidades}` en `resultado`.

> 🔁 **Qué cambia:** Antes contabas **ventas** por categoría. Ahora querés las **unidades vendidas** (la suma de `cantidad`) por categoría.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

unidades = {}

resultado = unidades

comprobar("py-g01")

## 17. El total que solo cuenta la última venta
`Debugging` · `Fácil`

Este loop debería dar el total facturado de todas las ventas, pero da un número demasiado chico. Encontrá el error.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

for v in ventas:
    total = 0
    total += v["cantidad"] * v["precio"]
resultado = total

comprobar("py-e05")

## 18. ¿Cuántas semanas hasta reponer?
`Código` · `Medio`

Un depósito tiene **120 unidades**. Cada semana el stock baja un **15%** (el nuevo stock es `int(stock * 0.85)`, que descarta decimales). ¿Cuántas semanas pasan hasta que el stock queda **por debajo de 30**? Guardá la cantidad de semanas en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

stock = 120
semanas = 0

resultado = semanas

comprobar("py-d11")

---
# 📌 Funciones: empaquetar lógica

## 19. Descuento con valor por defecto
`Código` · `Medio`

Escribí la función `aplicar_descuento(precio, porcentaje=10)` que devuelva el precio con el descuento aplicado, **redondeado a 2 decimales**. Si no se indica porcentaje, se usa 10.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

def aplicar_descuento(precio, porcentaje=10):
    pass

comprobar("py-d12")

## 20. Una función de resumen reutilizable
`Código` · `Medio`

Escribí `resumen(valores)` que devuelva una **tupla** `(mínimo, máximo, promedio)` con el promedio redondeado a 2 decimales. Después usala con `ventas_semana` y guardá el resultado en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

def resumen(valores):
    pass

resultado = resumen(ventas_semana)

comprobar("py-d13")

## 21. La función que 'calcula' pero no devuelve
`Debugging` · `Fácil`

La función muestra el número correcto en pantalla, pero `resultado` queda vacío. Corregila.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

def ingreso(venta):
    print(venta["cantidad"] * venta["precio"])

resultado = ingreso(ventas[0])

comprobar("py-e06")

## 22. La IA arma totales por cliente
`IA copiloto` · `Medio`

Pediste una función que calcule cuánto gastó cada cliente. La IA respondió esto. Leelo, ejecutalo y respondé las preguntas.

> 🤖 Prompt usado:
> Necesito una función en Python puro que reciba una lista de ventas (diccionarios con 'cliente', 'cantidad' y 'precio') y devuelva un diccionario cliente → total gastado. Explicá cada línea.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

def total_por_cliente(ventas):
    totales = {}
    for v in ventas:
        monto = v["cantidad"] * v["precio"]
        totales[v["cliente"]] = totales.get(v["cliente"], 0) + monto
    return totales

resultado = total_por_cliente(ventas)

comprobar("py-f02")

---
# 📌 Comprensiones

## 23. Ingresos de cada venta
`Código` · `Fácil`

Armá una lista con el **ingreso** (`cantidad × precio`) de cada venta de `ventas`, en el mismo orden. Guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("py-d14")

## 24. Solo las ventas grandes
`Código` · `Medio`

Obtené la lista de **nombres de producto** de las ventas cuyo ingreso (`cantidad × precio`) sea **mayor a 200**, en el orden en que aparecen. Guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("py-d15")

## 25. Transferencia: un diccionario con IVA
`Transferencia` · `Medio`

Creá un diccionario `{producto: precio_con_iva}` a partir de `precios`, con el IVA del 21% y redondeado a 2 decimales. Guardalo en `resultado`.

> 🔁 **Qué cambia:** Antes armaste una **lista** transformando cada venta. Ahora querés un **diccionario** transformando cada par del catálogo.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("py-g02")

---
# 📌 Errores y excepciones

## 26. Convertir sin caerse
`Código` · `Medio`

Convertí cada texto de `precios_texto` a `float`. Los que no se puedan convertir deben quedar como `None`. Guardá la lista resultante (mismo orden y largo) en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

precios_texto = ["1200", "25.5", "n/d", "45", "", "300.0"]

resultado = []

comprobar("py-d16")

## 27. El except que no atrapa nada
`Debugging` · `Medio`

Este código intenta manejar los precios inválidos, pero igual se cae. ¿Por qué?

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

precios_texto = ["1200", "n/d", "45"]
resultado = []
for t in precios_texto:
    try:
        resultado.append(float(t))
    except KeyError:
        resultado.append(None)

comprobar("py-e07")

---
# 📌 Módulos e imports

## 28. Media vs mediana con un pico
`Código` · `Medio`

Las lecturas válidas de un sensor tienen un pico extraño (85.3). Usando el módulo `statistics`, calculá la **media** (redondeada a 2) y la **mediana** de `validas`, y guardá `[media, mediana]` en `resultado`. Después mirá la diferencia: ¿cuál representa mejor la temperatura típica?

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

validas = [21.5, 22.1, 23.4, 22.8, 24.0, 85.3, 23.1]

resultado = 

comprobar("py-d17")

## 29. La IA usa fechas
`IA copiloto` · `Medio`

Querés saber cuántos días pasaron entre la primera y la última venta. La IA te dio este código con dos imports. Analizalo.

> 🤖 Prompt usado:
> Tengo una lista de ventas con fechas en texto 'AAAA-MM-DD'. ¿Cómo calculo cuántos días hay entre la primera y la última venta? Explicame qué módulo usás y por qué.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

from datetime import date

fechas = [date.fromisoformat(v["fecha"]) for v in ventas]
dias = (max(fechas) - min(fechas)).days
resultado = dias

comprobar("py-f03")

---
# 📌 Trabajar con archivos

## 30. Total desde un archivo CSV
`Código` · `Medio`

El archivo en `RUTA_VENTAS` tiene las columnas `fecha,producto,cantidad,precio`. Leelo con el módulo `csv` y calculá el total facturado (suma de cantidad × precio). Guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

import csv

total = 0
# abrí el archivo y recorré las filas

resultado = total

comprobar("py-d18")

## 31. Configuración en JSON
`Código` · `Fácil`

La configuración del análisis está en un archivo JSON (`RUTA_CONFIG`). Leela y calculá el precio final del `'Monitor'` (de `precios`) aplicando el IVA que indica la configuración. Redondeá a 2 decimales y guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

import json

resultado = 

comprobar("py-d19")

## 32. El CSV que no se deja sumar
`Debugging` · `Fácil`

Este código lee el CSV pero falla al calcular. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

import csv

total = 0
with open(RUTA_VENTAS, encoding="utf-8") as f:
    for fila in csv.DictReader(f):
        total += fila["cantidad"] * fila["precio"]
resultado = total

comprobar("py-e08")

---
# 📌 Leer código y debuggear

## 33. La lista ordenada que desapareció
`Debugging` · `Fácil`

Querés las ventas de la semana ordenadas de mayor a menor, pero `resultado` queda en `None`.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

ordenadas = ventas_semana.sort(reverse=True)
resultado = ordenadas

comprobar("py-e09")

---
# 📌 Python aplicado a datos

## 34. Top 3 productos por ingreso
`Código` · `Difícil`

Obtené el **top 3 de productos por ingreso total** (sumando `cantidad × precio` de todas sus ventas). Guardá una lista de tuplas `(producto, ingreso)` ordenada de mayor a menor en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

totales = {}

resultado = 

comprobar("py-d20")

## 35. La IA te propone pandas
`IA copiloto` · `Medio`

Le pediste a una IA las unidades por categoría y, en lugar del loop que vos hiciste, te propuso **pandas**. Comparalo con tu solución en Python puro.

> 🤖 Prompt usado:
> Tengo una lista de diccionarios 'ventas' con claves categoria y cantidad. Ya lo resolví con un for y un diccionario. ¿Cómo sería con pandas? Explicame la equivalencia con mi loop.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

import pandas as pd

df = pd.DataFrame(ventas)
resultado = df.groupby("categoria")["cantidad"].sum()

comprobar("py-f04")

---
# 📌 Proyecto: mini sistema de análisis de ventas

## 36. Fase 2 · Cargar y limpiar
`Código` · `Difícil`

Leé `RUTA_VENTAS_SUCIO` y construí `limpias`: una lista de diccionarios `{'producto': str, 'cantidad': int, 'precio': float}` solo con las filas **válidas** (cantidad convertible a int y mayor que 0). Contá las descartadas en `descartadas`. Guardá `limpias` en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

import csv

limpias = []
descartadas = 0

resultado = limpias

comprobar("py-p2")

## 37. Fase 3 · Calcular el reporte
`Código` · `Difícil`

Escribí la función `reporte(filas)` que reciba una lista de ventas limpias y devuelva un diccionario con:
- `'total'`: total facturado,
- `'por_producto'`: diccionario producto → ingreso,
- `'top'`: el producto de mayor ingreso.

Usala con las filas de `limpias` (ya cargadas en el código inicial) y guardá el resultado en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

limpias = [
    {"producto": "Laptop", "cantidad": 1, "precio": 1200.0},
    {"producto": "Mouse", "cantidad": 3, "precio": 25.5},
    {"producto": "Auriculares", "cantidad": 2, "precio": 80.0},
    {"producto": "Laptop", "cantidad": 2, "precio": 1200.0},
    {"producto": "Tablet", "cantidad": 1, "precio": 450.0},
]

def reporte(filas):
    pass

resultado = reporte(limpias)

comprobar("py-p3")

## 38. Fase 4 · Verificar: el promedio sospechoso
`Debugging` · `Medio`

Agregaste al reporte el **ticket promedio** (ingreso total ÷ cantidad de ventas), pero da un valor que no cierra. Encontrá el error verificando con un caso conocido.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

filas = [
    {"producto": "Laptop", "cantidad": 1, "precio": 1200.0},
    {"producto": "Mouse", "cantidad": 3, "precio": 25.5},
    {"producto": "Auriculares", "cantidad": 2, "precio": 80.0},
]

def ticket_promedio(filas):
    total = 0
    for f in filas:
        total += f["cantidad"] * f["precio"]
    return total / sum(f["cantidad"] for f in filas)

resultado = round(ticket_promedio(filas), 2)

comprobar("py-p4")

---
# 🏁 Fin del notebook
Volvé a la [plataforma](https://rosalesluciano.github.io/datalab/) para los ejercicios de reconocimiento, razonamiento y el modo **¿Sé esto?**.